<h1 style="color:#FFFFFF; background-color:#4C1D95; text-align:center; font-weight:bold; padding:17px 10px; border-radius:8px; margin-bottom:6px;">Week 6 — Day 4: Building &amp; Training a Network in Keras</h1>



<a id="toc"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">Table of Contents</h2>
<div style="border:1px solid #94A3B8; padding:15px 25px; border-radius:7px; line-height:1.9;">
<ol style="font-weight:700;">
<li><a href="#section0">0. Setup — Importing TensorFlow, Pandas &amp; Scikit-learn</a></li>
<li><a href="#section1">1. TensorFlow / Keras</a></li>
<li><a href="#section2">2. Building a Model with the Sequential API</a></li>
<li><a href="#section3">3. Compile, Train, Evaluate</a></li>
<li><a href="#section4">4. Batch Normalization and Dropout</a></li>
<li><a href="#section5">5. Common Mistakes to Avoid</a></li>
<li><a href="#section6">6. Quick Reference</a></li>
<li><a href="#section7">7. Hands-On Lab — Training a Neural Network</a></li>
<li><a href="#section8">8. Best Practices &amp; Reproducibility</a></li>
<li><a href="#section9">9. Summary — What I Learned Today</a></li>
<li><a href="#section10">10. Daily Stand-up</a></li>
</ol>
</div>


<a id="section0"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">0. Setup — Importing TensorFlow, Pandas &amp; Scikit-learn</h2>
<div style="border-left:5px solid #7C3AED; background-color:rgba(124,58,237,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Note:</b> Days 1–3 built the neuron, activations, loss, and backpropagation entirely by hand in NumPy — on the project's small 918-row <code>heart_cleaned.csv</code>. Today switches to a <b>production framework, Keras</b>, and deliberately switches to a much larger dataset: <b>253,680 patient records</b> from the CDC's BRFSS 2015 survey. Deep learning tends to shine only once there is real data volume to learn from — 918 rows was never enough to justify a neural network on its own, but 253,680 rows is exactly the kind of scale where it becomes worth testing.
</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

print("NumPy version     :", np.__version__)
print("Pandas version    :", pd.__version__)
print("TensorFlow version:", tf.__version__)
print("Keras version     :", keras.__version__)

<div style="border-left:5px solid #B45309; background-color:rgba(180,83,9,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Important:</b> <b>Dataset change from Days 1–3:</b> this notebook uses <code>heart_disease_health_indicators_BRFSS2015.csv</code> (253,680 rows, 22 columns, target <code>HeartDiseaseorAttack</code>), <i>not</i> the smaller <code>heart_cleaned.csv</code> from earlier in the week. The two datasets cover the same real-world problem — cardiac risk prediction — but this one has the row count deep learning actually needs. Because it is a different dataset, Day 4 establishes its <b>own fresh baseline</b> (Step 2 of the lab) rather than reusing Day 1's 87.50% figure directly.
</div>


<a id="section1"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">1. TensorFlow / Keras</h2>
You will not implement neural networks from scratch in production — frameworks handle the math. <b>TensorFlow</b> with its <b>Keras</b> API is the standard high-level framework: you describe the architecture layer by layer, and Keras handles forward propagation, backpropagation, and optimization automatically. (PyTorch is the main alternative, covered as needed; both are on the program's stack.)

<a id="section2"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">2. Building a Model with the Sequential API</h2>
The Keras <b>Sequential API</b> stacks layers in order. A <b>Dense</b> (fully connected) layer connects every input to every neuron. This example builds a network for binary classification:

In [ ]:
# Illustrative shape -- the real model is built with this project's actual n_features in Section 7.
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

example_model = Sequential([
    Dense(64, activation="relu", input_shape=(21,)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),   # binary output
])
example_model.summary()

<a id="section3"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">3. Compile, Train, Evaluate</h2>
The three-step Keras workflow mirrors the Scikit-learn API from Week 3. <code>compile()</code> sets the optimizer, loss, and metrics; <code>fit()</code> runs the training loop; <code>evaluate()</code> measures performance on the test set.

In [ ]:
# Illustrative call shape -- the real compile/fit/evaluate calls happen in Section 7.
example_model.compile(optimizer="adam",
                       loss="binary_crossentropy",
                       metrics=["accuracy"])

# history = model.fit(X_train, y_train,
#                      validation_data=(X_val, y_val),
#                      epochs=50, batch_size=32)
# model.evaluate(X_test, y_test)

print("The real training run happens in Section 7, on the actual project data.")

<div style="border-left:5px solid #7C3AED; background-color:rgba(124,58,237,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Note:</b> The <code>history</code> object records the loss and metric per epoch for both training and validation — plotting it is how you diagnose training, using the exact overfit/underfit reasoning from Week 4.
</div>


<a id="section4"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">4. Batch Normalization and Dropout</h2>
Two standard layers improve training and fight overfitting. <b>Batch normalization</b> normalizes each layer's inputs during training, making the network train faster and more stably. <b>Dropout</b> randomly "switches off" a fraction of neurons during each training step, forcing the network not to rely too heavily on any one path — a powerful regularizer, the deep-learning equivalent of the regularization from Week 4.

In [ ]:
from tensorflow.keras.layers import BatchNormalization, Dropout

example_model_v2 = Sequential([
    Dense(64, activation="relu", input_shape=(21,)),
    BatchNormalization(),
    Dropout(0.3),                    # drop 30% of neurons each step
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),
])
example_model_v2.summary()

<a id="section5"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">5. Common Mistakes to Avoid</h2>
<div style="border-left:5px solid #B45309; background-color:rgba(180,83,9,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Important:</b> <ul>
<li><b>Judging an imbalanced dataset by accuracy alone</b> — this dataset is roughly 90.6% / 9.4%, so a model that always predicts "no heart disease" would score over 90% accuracy while being useless; use F1-score, recall, and ROC-AUC, exactly as in Week 3.</li>
<li><b>Forgetting to scale features before training</b> — Keras networks train far more reliably on scaled inputs, just like the distance-based models from Week 4–5.</li>
<li><b>Not using a validation split</b> — without <code>validation_data</code>, there is no way to see overfitting happening during training.</li>
<li><b>Reading only the final epoch's numbers</b> — always plot the full loss/metric curves; a model that overfits late is invisible if you only check the last epoch.</li>
<li><b>Ignoring class imbalance during training</b> — on a 90/10 split like this one, consider <code>class_weight</code> so the rarer positive class isn't drowned out.</li>
</ul>
</div>


<a id="section6"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">6. Quick Reference </h2>


<div style="border:1px solid #ccc; border-radius:6px; overflow:hidden;">
<table style="width:100%; border-collapse:collapse;">
<tr style="background-color:#4C1D95; color:#FFFFFF;"><th style="padding:9px; text-align:left;">Task</th><th style="padding:9px; text-align:left;">Code</th></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Import Keras</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>from tensorflow.keras import Sequential</code><br><code>from tensorflow.keras.layers import Dense</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Build a model</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>Sequential([Dense(64, activation="relu", input_shape=(n,)), Dense(1, activation="sigmoid")])</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Compile</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Train</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32)</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Evaluate</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>model.evaluate(X_test, y_test)</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Predict probabilities</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>model.predict(X_test)</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Batch normalization</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>from tensorflow.keras.layers import BatchNormalization</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Dropout</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>Dropout(0.3)</code>  # drop 30% of neurons</td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Class weighting (imbalance)</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>model.fit(..., class_weight={0: w0, 1: w1})</code></td></tr>
</table>
</div>


<a id="section7"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">7. Hands-On Lab — Training a Neural Network</h2>
<div style="border-left:5px solid #0E7490; background-color:rgba(14,116,144,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Goal:</b> Build a Keras Sequential network for the Cardiac Patient Monitoring System's binary classification task, train it on the full 253,680-row BRFSS dataset, diagnose its fit, then improve and re-evaluate it against a freshly established baseline.
</div>


<h3 style="color:#FFFFFF; background-color:#5B21B6; opacity:0.92; display:inline-block; padding:6px 12px; border-radius:6px; font-weight:bold;">7.0 Loading the dataset and establishing a fresh baseline</h3>
This step loads <code>heart_disease_health_indicators_BRFSS2015.csv</code> and trains a quick Logistic Regression baseline <i>on this specific dataset</i>, so the neural network below is judged fairly against a baseline from the same data — not against Day 1's baseline from the smaller <code>heart_cleaned.csv</code>.

In [ ]:
df = pd.read_csv("heart_disease_health_indicators_BRFSS2015.csv")

print("Shape:", df.shape)
print()
print("Target balance:")
print(df["HeartDiseaseorAttack"].value_counts())
print((df["HeartDiseaseorAttack"].value_counts(normalize=True) * 100).round(2))

df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["HeartDiseaseorAttack"])
y = df["HeartDiseaseorAttack"]
n_features = X.shape[1]
print("Number of input features:", n_features)

# Three-way split: 60% train / 20% validation / 20% test, stratified to preserve class balance
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Scale AFTER splitting, fit on train only -- the Week 4 leakage rule
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape[0]:,} | Val: {X_val_scaled.shape[0]:,} | Test: {X_test_scaled.shape[0]:,}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
baseline.fit(X_train_scaled, y_train)

baseline_pred = baseline.predict(X_test_scaled)
baseline_proba = baseline.predict_proba(X_test_scaled)[:, 1]

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)
baseline_auc = roc_auc_score(y_test, baseline_proba)

print(f"Baseline (Logistic Regression) on this dataset:")
print(f"  Accuracy: {baseline_accuracy:.4f}")
print(f"  F1 Score: {baseline_f1:.4f}")
print(f"  ROC-AUC : {baseline_auc:.4f}")

<div style="border-left:5px solid #7C3AED; background-color:rgba(124,58,237,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Note:</b> <code>class_weight="balanced"</code> is used for the baseline precisely because of the 90.6%/9.4% imbalance flagged in Section 5 — without it, Logistic Regression would lean heavily toward always predicting "no heart disease."
</div>


<h3 style="color:#FFFFFF; background-color:#5B21B6; opacity:0.92; display:inline-block; padding:6px 12px; border-radius:6px; font-weight:bold;">Step 1 — Build a Keras Sequential network appropriate for the Phase 3 project task</h3>


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),   # binary output: probability of HeartDiseaseorAttack
])

model.summary()

<h3 style="color:#FFFFFF; background-color:#5B21B6; opacity:0.92; display:inline-block; padding:6px 12px; border-radius:6px; font-weight:bold;">Step 2 — Compile with Adam and the right loss, then train with a validation split</h3>


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Class weights so the network doesn't ignore the rarer positive class during training
class_weight = {0: 1.0, 1: (y_train == 0).sum() / (y_train == 1).sum()}
print("Class weights used during training:", class_weight)

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=256,
    class_weight=class_weight,
    verbose=0,
)
print("Training complete.")

<h3 style="color:#FFFFFF; background-color:#5B21B6; opacity:0.92; display:inline-block; padding:6px 12px; border-radius:6px; font-weight:bold;">Step 3 — Plot training vs. validation loss and accuracy, and diagnose the fit</h3>


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history.history["loss"], color="#0072B2", linewidth=2.2, label="Training loss")
axes[0].plot(history.history["val_loss"], color="#E69F00", linewidth=2.2, linestyle="--", label="Validation loss")
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history.history["accuracy"], color="#0072B2", linewidth=2.2, label="Training accuracy")
axes[1].plot(history.history["val_accuracy"], color="#E69F00", linewidth=2.2, linestyle="--", label="Validation accuracy")
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

<div style="border-left:5px solid #B45309; background-color:rgba(180,83,9,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Important:</b> <b>Diagnose the fit (Week 4 reasoning):</b> if training and validation loss track closely together and both keep falling, the model is fitting well. If training loss keeps dropping while validation loss flattens or rises, that gap is <b>overfitting</b> — exactly the pattern Batch Normalization and Dropout (Section 4) are used to fix, tested next in Step 4.
</div>


<h3 style="color:#FFFFFF; background-color:#5B21B6; opacity:0.92; display:inline-block; padding:6px 12px; border-radius:6px; font-weight:bold;">Step 4 — Add dropout and/or batch normalization and compare the new loss curves</h3>


In [ ]:
from tensorflow.keras.layers import BatchNormalization, Dropout

model_v2 = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])

model_v2.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history_v2 = model_v2.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=256,
    class_weight=class_weight,
    verbose=0,
)
print("Training complete for the regularized model.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history.history["val_loss"], color="#E69F00", linewidth=2.2, linestyle="--", label="Plain network")
axes[0].plot(history_v2.history["val_loss"], color="#009E73", linewidth=2.2, label="With BatchNorm + Dropout")
axes[0].set_title("Validation Loss — Plain vs. Regularized")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history.history["val_accuracy"], color="#E69F00", linewidth=2.2, linestyle="--", label="Plain network")
axes[1].plot(history_v2.history["val_accuracy"], color="#009E73", linewidth=2.2, label="With BatchNorm + Dropout")
axes[1].set_title("Validation Accuracy — Plain vs. Regularized")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

<h3 style="color:#FFFFFF; background-color:#5B21B6; opacity:0.92; display:inline-block; padding:6px 12px; border-radius:6px; font-weight:bold;">Step 5 — Evaluate on the test set and compare the score to the baseline</h3>


In [ ]:
from sklearn.metrics import classification_report

test_proba = model_v2.predict(X_test_scaled, verbose=0).ravel()
test_pred = (test_proba >= 0.5).astype(int)

nn_accuracy = accuracy_score(y_test, test_pred)
nn_f1 = f1_score(y_test, test_pred)
nn_auc = roc_auc_score(y_test, test_proba)

comparison = pd.DataFrame({
    "Model": ["Logistic Regression (baseline)", "Keras Neural Network (regularized)"],
    "Accuracy": [round(baseline_accuracy, 4), round(nn_accuracy, 4)],
    "F1 Score": [round(baseline_f1, 4), round(nn_f1, 4)],
    "ROC-AUC": [round(baseline_auc, 4), round(nn_auc, 4)],
})
comparison

In [ ]:
print(classification_report(y_test, test_pred, target_names=["No disease", "Heart disease/attack"]))

<div style="border-left:5px solid #0E7490; background-color:rgba(14,116,144,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Goal:</b> The table above is the honest, apples-to-apples comparison for this dataset: both models were trained on the identical train/validation/test split, with the identical class-weighting strategy, and evaluated on the same held-out test set. Whichever model wins on F1-score and ROC-AUC — not raw accuracy alone, given the 90.6%/9.4% imbalance — is the one that earns a place in the Cardiac Patient Monitoring System going forward.
</div>


<a id="section8"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">8. Best Practices &amp; Reproducibility</h2>
<div style="border-left:5px solid #B45309; background-color:rgba(180,83,9,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Important:</b> <ul>
<li>Fix <code>np.random.seed(42)</code> and <code>tf.random.set_seed(42)</code> for reproducible weight initialization and training.</li>
<li>Always use a three-way split (train/validation/test) and scale features fit on the training set only.</li>
<li>Plot the full <code>history</code> curves, not just the final epoch's numbers, to catch overfitting.</li>
<li>On imbalanced data, use <code>class_weight</code> and evaluate with F1/ROC-AUC, never accuracy alone.</li>
<li>When switching datasets mid-project (as today), establish a <b>fresh baseline on the new dataset</b> rather than comparing across two different data sources.</li>
</ul>
</div>


<a id="section9"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">9. Summary — What I Learned Today</h2>
<div style="border-left:5px solid #C2410C; background-color:rgba(194,65,12,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Tip:</b> <ul>
<li><b>Keras</b> is the standard high-level framework — describe layers, let it handle forward propagation, backpropagation, and optimization.</li>
<li>The <code>Sequential</code> API stacks <code>Dense</code> layers; <code>compile() &rarr; fit() &rarr; evaluate()</code> mirrors the Week 3 Scikit-learn workflow.</li>
<li>The training <code>history</code> object is how overfitting and underfitting are diagnosed in deep learning, using the exact reasoning from Week 4.</li>
<li><b>Batch normalization</b> and <b>dropout</b> are the deep-learning equivalents of regularization.</li>
<li>Deep learning is worth testing once there is enough data — today's 253,680-row BRFSS dataset is the scale where a neural network has a real chance to add value over classical ML.</li>
<li>The Keras model was evaluated fairly against a freshly trained Logistic Regression baseline on the <i>same</i> data and split — never against a score from a different dataset.</li>
</ul>
</div>
<div style="border-left:5px solid #7C3AED; background-color:rgba(124,58,237,0.11); padding:11px 15px; margin:12px 0; border-radius:5px; color:inherit; line-height:1.55;">
<b>Note:</b> <b>Where this leads next:</b> Day 5 wraps up Sprint 1: documenting the final model choice, updating the project README with the real results, and preparing the repository for mentor review.
</div>


<a id="section10"></a>
<h2 style="color:#FFFFFF; background-color:#5B21B6; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">10. Daily Stand-up</h2>
<div style="border:1px solid #ccc; border-radius:6px; overflow:hidden;">
<table style="width:100%; border-collapse:collapse;">
<tr style="background-color:#4C1D95; color:#FFFFFF;"><th style="padding:9px; text-align:left;">Question</th><th style="padding:9px; text-align:left;">Answer</th></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">What did I complete today?</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Built, compiled, and trained a Keras Sequential neural network for the Cardiac Patient Monitoring System, on the full 253,680-row BRFSS dataset; established a fresh Logistic Regression baseline on this same dataset; diagnosed the plain network's fit from its loss/accuracy curves; added Batch Normalization and Dropout and compared the regularized model against both the plain network and the baseline on the held-out test set.</td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">What's next?</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Day 5: finalize the Sprint 1 deliverable — document the chosen model and its results, update the project README, and prepare for mentor review.</td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Any blockers?</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">None. heart_disease_health_indicators_BRFSS2015.csv loaded correctly from the same folder as this notebook.</td></tr>
</table>
</div>
